# SquatTrainer Project Workflow

This notebook is a planning document for building a deep learning based squat posture trainer from scratch. It does not contain implementation code yet. Its purpose is to organize the full project into phases, identify the best technologies to use, and clarify what each stage needs to produce before we move on.

## 1. Project Goal and Success Criteria

### Main Goal
Build a system that takes a squat video or webcam stream and predicts whether the user is performing the movement with correct or incorrect form, while also highlighting important mistake categories and providing interpretable feedback.

### Core Outputs
- Binary prediction: correct vs incorrect squat form.
- Optional multi-label mistake prediction: for example knees collapsing inward, shallow depth, excessive forward lean, heels lifting, or loss of back neutrality.
- Real-time visual feedback overlay for demonstration.

### Evaluation Targets
- Good accuracy on held-out people.
- Reasonable robustness to different camera angles, lighting, and clothing.
- Stable predictions across time rather than frame-by-frame flickering.
- Fast enough inference for a live webcam demo on a laptop.

## 2. What the Course Material Suggests We Should Use

After comparing the class notebooks, the best fit for this project is a **PyTorch-based deep learning pipeline built on body keypoints over time**.

### Lessons from the Course Notebooks
- `knn.ipynb`: useful for understanding simple classification baselines, but not strong enough for temporal movement analysis in realistic video.
- `linear_regression.ipynb` and `logistic_regression.ipynb`: good for baseline experiments on engineered features, but too limited for complex posture dynamics.
- `neural_network.ipynb` and `classification_nn.ipynb`: show that learned nonlinear models are much better when feature relationships are complex.
- `assignment5.ipynb`: the strongest match for implementation style because it introduces **PyTorch**, training loops, convolutional models, and modern deep learning workflow.
- `assignment6.ipynb`: useful because it reinforces **dense visual prediction**, pretrained backbones, and structured vision pipelines, even though segmentation itself is not the core task here.

### Recommended Technologies
- **PyTorch** for model training, experimentation, and deployment of the temporal classifier.
- **A pretrained pose estimator** to extract body joints from each frame instead of learning directly from raw pixels from scratch.
- **OpenCV** for video loading, frame extraction, webcam capture, and overlay rendering.
- **NumPy / Pandas / Matplotlib** for preprocessing, analysis, and visualization.

### Best Modeling Strategy
The strongest overall design is:

**Video frames -> pose keypoints -> normalized pose sequence -> temporal neural network -> form prediction + feedback**

This is better than training on raw images alone because squat quality depends on joint geometry and motion across time. Keypoints are lower-dimensional, more interpretable, and usually more robust to background clutter and lighting changes.

## 3. End-to-End System Design

### Input
- Short squat video clips.
- Live webcam stream.

### Intermediate Representation
- Per-frame 2D body keypoints.
- Optional confidence scores for each keypoint.
- Normalized joint coordinates centered around hips or torso.
- Temporal windows representing one squat or a short sequence of movement.

### Output
- Form quality label.
- Mistake category labels.
- Confidence score.
- Visual explanation overlay on the video stream.

## 4. Phase 1: Define Labels and Movement Criteria

Before training anything, we need to define what counts as good and bad form in a consistent way.

### Tasks
- Decide the exact exercise scope for the first version: bodyweight squat only, or squat plus other exercises later.
- Define the class labels.
- Define mistake categories.
- Write annotation rules so each example is labeled consistently.

### Suggested Label Structure
- `correct_form`
- `incorrect_form`
- `knees_inward`
- `insufficient_depth`
- `excessive_forward_lean`
- `heels_lifting`
- `uncontrolled_descent_or_ascent`

### Deliverable for This Phase
A written rubric that defines each label clearly enough that two different people would annotate the same squat clip similarly.

## 5. Phase 2: Data Collection and Dataset Construction

This phase is critical because the project will likely depend on self-collected squat footage.

### Data We Need
- Videos of correct form.
- Videos of common incorrect form patterns.
- Multiple people with different heights, builds, clothing styles, and experience levels.
- Different camera distances and angles, especially side and front views.
- Different lighting and room backgrounds.

### Dataset Organization
- Raw video folder.
- Metadata file for subject ID, camera angle, lighting condition, and label.
- Split into train, validation, and test sets.
- Ensure person-level separation so the test set includes unseen people.

### Deliverable for This Phase
A clean, labeled dataset with documented train/validation/test splits and enough diversity to test generalization.

## 6. Phase 3: Pose Extraction and Preprocessing Pipeline

This is where the raw videos become structured learning inputs.

### Tasks
- Run a pretrained pose estimator on every frame.
- Store keypoints and confidence values.
- Handle missing or low-confidence joints.
- Normalize pose coordinates to reduce dependence on camera distance and body size.
- Segment videos into temporal windows or individual squat repetitions.

### Why This Phase Matters
The pose representation should capture movement while discarding irrelevant details like background clutter, wall color, and clothing texture.

### Deliverable for This Phase
A reusable preprocessing pipeline that converts each video into a consistent keypoint sequence representation ready for model training.

## 7. Phase 4: Baselines Before the Main Deep Model

The course notebooks suggest that we should build simple baselines first before training the full temporal system.

### Baseline Options Inspired by the Class
- **KNN baseline** on engineered pose features for a very simple reference point.
- **Logistic regression baseline** on pose angles and distances.
- **Small fully connected neural network** on flattened keypoint features.

### Purpose of Baselines
- Check whether the labels are learnable.
- Establish minimum performance.
- Catch preprocessing mistakes early.
- Provide a comparison point when the temporal model is added.

### Deliverable for This Phase
A table of baseline models and their validation performance so we know whether the deeper model is truly improving results.

## 8. Phase 5: Main Deep Learning Model

This is the core modeling phase.

### Recommended Main Model
Use a **temporal neural network in PyTorch** that takes a sequence of normalized body keypoints and predicts squat quality over time.

### Strong Candidate Architectures
- **1D temporal CNN** over keypoint sequences.
- **LSTM or GRU** over keypoint sequences.
- Optional future extension: hybrid model that combines pose features with image features.

### Best First Choice
A **1D temporal CNN or small LSTM in PyTorch** is likely the best first implementation because it matches the course deep learning progression, is easier to train than a large video model, and directly models movement over time.

### Inputs to the Model
- Joint coordinates.
- Derived pose angles.
- Optional joint velocities across frames.
- Optional keypoint confidence scores.

### Outputs from the Model
- Binary form classification.
- Optional multi-label mistake detection.
- Optional per-frame or per-repetition confidence.

### Deliverable for This Phase
A trainable PyTorch model definition and a clear experiment plan for architecture comparison.

## 9. Phase 6: Training Strategy and Validation Plan

### Training Workflow
- Build PyTorch datasets and dataloaders.
- Train on the training split.
- Tune hyperparameters on the validation split.
- Keep the test set untouched until final evaluation.

### Important Hyperparameters
- Learning rate.
- Batch size.
- Sequence length.
- Hidden dimension or channel count.
- Weight decay.
- Dropout.

### Metrics
- Accuracy.
- Precision, recall, and F1 score.
- Confusion matrix.
- Stability across consecutive frames.
- Generalization across unseen people and angles.

### Deliverable for This Phase
A reliable training pipeline with reproducible experiments and evaluation metrics that match the project goals.

## 10. Phase 7: Error Analysis and Model Improvement

Once the first model works, we need to understand where it fails.

### Questions to Investigate
- Does performance drop for certain camera angles?
- Does the model confuse minor mistakes with dangerous mistakes?
- Does it fail more often on certain body types or clothing styles?
- Are predictions unstable near the top or bottom of the squat?

### Possible Improvements
- Better labeling consistency.
- More diverse training videos.
- Better normalization of pose sequences.
- Temporal smoothing or sequence-level prediction.
- Multi-task learning for form plus mistake categories.

### Deliverable for This Phase
An organized error analysis report that explains the most common failure cases and what changes improve them.

## 11. Phase 8: Real-Time Demo and Interactive Feedback

The final system should show that the project works in a practical setting rather than only on saved videos.

### Demo Components
- Webcam capture.
- Real-time pose extraction.
- Sliding temporal window for inference.
- Overlay with predicted form status and mistake messages.
- Optional repetition counter and confidence display.

### Real-Time Constraints
- Keep latency low enough for useful feedback.
- Avoid prediction flicker by smoothing outputs across time.
- Make feedback interpretable and not overwhelming.

### Deliverable for This Phase
A live demonstration where a user performs a squat in front of a webcam and receives stable posture feedback.

## 12. Final Recommended Technology Stack

### Best Overall Stack
- **PyTorch** for the learning model.
- **Pretrained pose estimation** for extracting body joints.
- **Temporal sequence model** on keypoints for form classification.
- **OpenCV** for video processing and webcam demo.
- **Matplotlib / NumPy / Pandas** for analysis and experiments.

### Why This Is the Best Choice
- It uses the strongest deep learning concepts from the later class notebooks.
- It is more appropriate for motion and posture than KNN or linear classifiers.
- It is more data-efficient than raw video training from scratch.
- It is easier to explain and visualize for a final class project.
- It supports both offline evaluation and a real-time live demo.

### Not Recommended as the Main Approach
- KNN alone.
- Linear or logistic regression alone.
- Training a large raw-video model from scratch with limited data.
- Using segmentation as the main task, since posture depends more directly on joints and motion than pixel masks.

## 13. Suggested Implementation Order

1. Finalize labels and annotation rules.
2. Collect and organize squat videos.
3. Build the pose extraction and normalization pipeline.
4. Train simple baselines on pose features.
5. Train the main PyTorch temporal model.
6. Evaluate on unseen people and camera angles.
7. Perform error analysis and refine the pipeline.
8. Build the webcam demo and visual overlay.
9. Prepare final report, figures, and presentation results.

This order gives the project a clean progression from data to modeling to deployment, while keeping the work aligned with the material covered in class.